# Example API Surface

Canonical API/runtime routing example notebook for the public `api` facade.

## Scope

This notebook is the canonical example surface for `example_api_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import io
import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_api_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_api_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view. Canonical retained execution in this repo state is CPU-oriented, but the notebook calling pattern remains CPU/GPU portable and explicitly parameterized for `float32` and `float64`.

In [ ]:
SUPPORTED_JAX_MODES = ('cpu', 'gpu')
SUPPORTED_JAX_DTYPES = ('float32', 'float64')
if JAX_MODE not in SUPPORTED_JAX_MODES:
    raise ValueError(f'Unsupported JAX_MODE: {JAX_MODE}')
if JAX_DTYPE not in SUPPORTED_JAX_DTYPES:
    raise ValueError(f'Unsupported JAX_DTYPE: {JAX_DTYPE}')
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('supported_jax_modes:', SUPPORTED_JAX_MODES)
print('supported_jax_dtypes:', SUPPORTED_JAX_DTYPES)
print('validation_slice:', 'cpu_current__gpu_portable_contract')
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)
runtime_payload = json.loads(runtime.stdout)
(EXAMPLE_OUTPUT_ROOT / f'runtime_{JAX_MODE}.json').write_text(json.dumps(runtime_payload, indent=2) + '\n', encoding='utf-8')

## Object/Input Construction

Build representative scalar, interval, and matrix inputs that exercise the routed API.

In [ ]:
import jax.numpy as jnp
from arbplusjax import acb_core, api, double_interval as di

x = jnp.asarray(0.5, dtype=jnp.float64)
y = jnp.asarray(2.0, dtype=jnp.float64)
s = jnp.asarray(2.5, dtype=jnp.float64)
z = jnp.asarray(1.0, dtype=jnp.float64)
a_mid = jnp.array([[4.0, 1.0], [1.0, 3.0]], dtype=jnp.float64)
rhs_mid = jnp.array([[1.0], [2.0]], dtype=jnp.float64)
a = di.interval(a_mid, a_mid)
rhs = di.interval(rhs_mid, rhs_mid)
c_mid = jnp.array([[4.0 + 0.0j, 1.0 + 1.0j], [1.0 - 1.0j, 5.0 + 0.0j]], dtype=jnp.complex128)
c_rhs_mid = jnp.array([[1.0 + 0.5j], [2.0 - 0.25j]], dtype=jnp.complex128)
c_a = acb_core.acb_box(di.interval(jnp.real(c_mid), jnp.real(c_mid)), di.interval(jnp.imag(c_mid), jnp.imag(c_mid)))
c_rhs = acb_core.acb_box(di.interval(jnp.real(c_rhs_mid), jnp.real(c_rhs_mid)), di.interval(jnp.imag(c_rhs_mid), jnp.imag(c_rhs_mid)))

## Direct Usage

Compare direct routed `evaluate()` calls against the explicit public entrypoints.

In [ ]:
api_results = {
    'besselk_routed': api.evaluate('besselk', x, y, implementation='cuda_besselk', value_kind='real'),
    'incgamma_direct': api.incomplete_gamma_upper(s, z, method='quadrature', samples_per_panel=8, max_panels=16),
    'incgamma_routed': api.evaluate('incomplete_gamma_upper', s, z, method='quadrature', method_params={'samples_per_panel': 8, 'max_panels': 16}),
    'arb_mat_solve_routed': api.evaluate('arb_mat_solve', a, rhs, mode='basic', value_kind='real_matrix', dtype='float64'),
    'acb_mat_solve_routed': api.evaluate('acb_mat_solve', c_a, c_rhs, mode='basic', value_kind='complex_matrix', dtype='float64'),
}
display(api_results)

## Production Pattern

For routed library use, favor explicit `evaluate()` or pre-bound batch entrypoints with fixed `dtype`, `mode`, and `pad_to` so service calls do not constantly recompile on batch-length drift. Keep matrix plans cached when an operation supports prepare/apply separation.

In [ ]:
real_batch = jnp.asarray([0.5, 1.0, 1.5, 2.0, 2.5], dtype=jnp.float64)
gamma_bound = api.bind_point_batch('incomplete_gamma_upper', dtype='float64', pad_to=8, method='quadrature', regularized=True)
solve_bound = api.bind_interval_batch('arb_mat_solve', mode='basic', dtype='float64', pad_to=4, prec_bits=53)
cached_plan = api.eval_point('arb_mat_matvec_cached_prepare', a)
api_service_results = {
    'gamma_bound': gamma_bound(real_batch, jnp.asarray([1.0, 1.1, 1.2, 1.3, 1.4], dtype=jnp.float64)),
    'solve_bound': solve_bound(a, rhs),
    'cached_matvec': api.eval_point('arb_mat_matvec_cached_apply', cached_plan, rhs),
}
display(api_service_results)

## Extending Benchmarks

To extend routed benchmarks, add the target operation or implementation branch in `benchmark_api_surface.py`, `benchmark_special_function_service_api.py`, or `benchmark_matrix_service_api.py`, depending on whether the concern is generic API routing, special-function services, or matrix services.

## Fast JAX Point Pattern

The routed API should still show the compiled point-batch path explicitly. Use `bind_point_batch_jit()` when a routed function is going to be called repeatedly in a point-mode service loop.

In [ ]:
import jax
jit_gamma = api.bind_point_batch_jit('incomplete_gamma_upper', dtype='float64', pad_to=8, method='quadrature', regularized=True)
jit_out = jit_gamma(real_batch, jnp.asarray([1.0, 1.1, 1.2, 1.3, 1.4], dtype=jnp.float64))
vmap_out = jax.vmap(lambda s_i, z_i: api.evaluate('incomplete_gamma_upper', s_i, z_i, method='quadrature', regularized=True))(real_batch, jnp.asarray([1.0, 1.1, 1.2, 1.3, 1.4], dtype=jnp.float64))
display({'jit_shape': jit_out.shape, 'jit_matches_vmap': bool(jnp.allclose(jit_out, vmap_out, rtol=1e-6, atol=1e-6))})

## AD Product Pattern

The routed API should demonstrate AD through the same public metadata-aware entrypoints used in product code. This section differentiates a routed special-function call and plots primal and gradient behavior together.

In [ ]:
import jax
sweep = jnp.linspace(0.25, 1.5, 24, dtype=jnp.float64)
def routed_loss(zv):
    return jnp.sum(api.evaluate('incomplete_gamma_upper', jnp.asarray(2.5, dtype=jnp.float64), zv, method='quadrature', method_params={'samples_per_panel': 8, 'max_panels': 16}))
grad_vals = jax.grad(routed_loss)(sweep)
primal_vals = api.bind_point_batch('incomplete_gamma_upper', dtype='float64', pad_to=32, method='quadrature')(jnp.full_like(sweep, 2.5), sweep)
ad_df = pd.DataFrame({'z': np.asarray(sweep), 'primal': np.asarray(primal_vals), 'grad': np.asarray(grad_vals)})
display(ad_df.head())
ax = ad_df.plot(x='z', y=['primal', 'grad'], figsize=(8, 4), title='API Routed AD Validation')
ax.set_ylabel('value')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Parameter/Value Sweeps

Sweep the official API benchmark over representative routed cases.

In [ ]:
api_report = EXAMPLE_OUTPUT_ROOT / f'api_surface_{JAX_MODE}.json'
run([PYTHON, 'benchmarks/benchmark_api_surface.py', '--warmup', '1', '--runs', '3', '--output', str(api_report)])
api_payload = json.loads(api_report.read_text())
api_df = pd.DataFrame(api_payload['records'])
display(api_df[['operation', 'implementation', 'cold_time_s', 'warm_time_s', 'recompile_time_s']])

## Validation Summary

Run the API metadata and selection contract tests that back the routed public surface.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_api_metadata.py',
    'tests/test_api_selection_contracts.py',
    'tests/test_core_scalar_api_contracts.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
(EXAMPLE_OUTPUT_ROOT / f'pytest_{JAX_MODE}.txt').write_text(tests.stdout + ('\n' + tests.stderr if tests.stderr else ''), encoding='utf-8')

## Benchmark Summary

Summarize the official API benchmark artifacts emitted by the routed benchmark script.

In [ ]:
summary = api_df.groupby(['operation', 'implementation'])[['cold_time_s', 'warm_time_s', 'recompile_time_s']].mean(numeric_only=True)
summary.reset_index().to_csv(EXAMPLE_OUTPUT_ROOT / f'api_benchmark_summary_{JAX_MODE}.csv', index=False)
display(summary)

## Comparison Summary

The API layer is a routing surface rather than a separate numerical backend. Its comparison story is direct-vs-routed overhead, plus the downstream scalar/matrix comparison layers.

In [ ]:
print('direct-vs-routed overhead rows:')
display(api_df[['operation', 'implementation', 'warm_time_s']].sort_values(['operation', 'implementation']))

## Diagnostics

Run the existing matrix diagnostics entrypoint to capture compile, steady-state, and recompile behavior for representative routed matrix surfaces.

In [ ]:
diag_report = EXAMPLE_OUTPUT_ROOT / f'api_matrix_diagnostics_{JAX_MODE}.json'
run([PYTHON, 'benchmarks/benchmark_matrix_stack_diagnostics.py', '--n', '4', '--repeats', '2', '--output', str(diag_report)])
diag_df = pd.DataFrame(json.loads(diag_report.read_text()))
diag_df.to_csv(EXAMPLE_OUTPUT_ROOT / f'api_matrix_diagnostics_{JAX_MODE}.csv', index=False)
display(diag_df[['name', 'compile_ms', 'steady_ms_median', 'recompile_new_shape_ms', 'peak_rss_delta_mb']])

## Plots

Plot cold/warm/recompile timing by operation and implementation.

In [ ]:
pivot = api_df.pivot(index='operation', columns='implementation', values='warm_time_s')
ax = pivot.plot(kind='bar', figsize=(10, 4), title='API Warm Time by Operation')
ax.set_ylabel('warm_time_s')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'api_warm_time_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Optional Diagnostics

For compile/memory diagnostics beyond the API benchmark, use `benchmark_matrix_stack_diagnostics.py` or the JAX diagnostics helpers explicitly.

In [ ]:
summary_lines = [
    f'# Example API Surface Summary ({JAX_MODE})',
    '',
    f'- python: `{PYTHON}`',
    f'- backend: `{runtime_payload["platform"]}`',
    f'- api_rows: `{len(api_df)}`',
    f'- diagnostics_rows: `{len(diag_df)}`',
    '',
    '## Routed Operations',
    '',
]
for row in summary.reset_index().to_dict(orient='records'):
    summary_lines.append(f"- `{row['operation']}` / `{row['implementation']}`: warm={row['warm_time_s']:.6g}s, cold={row['cold_time_s']:.6g}s, recompile={row['recompile_time_s']:.6g}s")
summary_lines.extend(['', '## Diagnostics Cases', ''])
for row in diag_df.to_dict(orient='records'):
    summary_lines.append(f"- `{row['name']}`: compile_ms={row['compile_ms']:.6g}, steady_ms_median={row['steady_ms_median']:.6g}, recompile_new_shape_ms={row['recompile_new_shape_ms']:.6g}")
(EXAMPLE_OUTPUT_ROOT / f'summary_{JAX_MODE}.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
display('\n'.join(summary_lines[:16]))